# Lentils × Dinomaly — Concrete (learnable band selector) training tutorial

Instead of hand-picking bands (RGB) or a fixed CIR combination, **learn** which of the 61
VNIR bands feed Dinomaly's 3-channel input. `ConcreteChannelMixer` uses a Concrete
(Gumbel-softmax) relaxation with a temperature `tau` annealed from 10 → 0.1 across training,
so the soft mixture sharpens into a near-hard band selection; a **distinctness** regulariser
keeps the 3 output channels from collapsing onto the same band.

The selector is trained **jointly** with the Dinomaly bottleneck + decoder (the DINOv2
encoder stays frozen). Modeled on `examples/train_dinomaly_concrete_joint_multifile.py`.

Sibling notebooks — `lentils_rgb_train_tutorial.ipynb`, `lentils_cir_train_tutorial.ipynb`,
and `lentils_inference_tutorial.ipynb` (evaluate + per-class AUROC).

> **Prerequisites**
> - Run from the repo's `notebooks/lentils_anomaly/` folder (imports the sibling `utils.py`
>   and the repo's `examples/plugins.yaml`).
> - Env: cuvis-ai ≥ 0.10, cuvis-ai-core ≥ 0.10, cuvis-ai-schemas ≥ 0.7,
>   cuvis-ai-dataloader ≥ 0.3, anomalib 2.1, a CUDA GPU.
> - `LENTILS_DATA_SOURCE=local` (default) reads the on-server NPZ (correct baked masks); `hf`
>   is blocked on a converter fix (see the RGB tutorial's note).
> - Knobs are env-overridable (`LENTILS_MAX_EPOCHS`, `LENTILS_IMAGE_SIZE`, `LENTILS_SMOKE_LIMIT`).
>   **Note:** the `tau` anneal schedule depends on `MAX_EPOCHS`, so a 1-epoch smoke keeps the
>   selector soft — use the full run for a meaningful learned selection.

## 1 · Dataset overview

61-channel VNIR (430–910 nm) cubes of lentils on a conveyor; foreign objects (stones, metal
shards, paper, rubber, insects) are the anomalies — 8 COCO categories (0 = normal). Published
at `cubert-gmbh/XMR_Industrial_Foreign_Object_Detection_Lentils`.

**Dinomaly (train-on-normals) split** — `splits_dinomaly.csv`: train 308 (normal) / val 148 /
test 180 / adaclip_train 500 (held out). Dinomaly trains on the 308 normal frames only.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import torch
import pytorch_lightning as pl

sys.path.insert(0, str(Path('.').resolve()))
import utils
from utils import LENTILS_CATEGORIES, resolve_config, resolve_splits_csv

config = resolve_config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:        ', DEVICE)
print('Dataset source:', config['data_source'])
print('Selector:       Concrete learnable band mixer (61 -> 3 channels)')

# --- Tutorial knobs (env-overridable) ------------------------------------
MAX_EPOCHS  = int(os.environ.get('LENTILS_MAX_EPOCHS', '1'))    # bump to 50 for the full run
IMAGE_SIZE  = int(os.environ.get('LENTILS_IMAGE_SIZE', '448'))  # square side, multiple of 14
SMOKE_LIMIT = int(os.environ.get('LENTILS_SMOKE_LIMIT', '0'))   # 0 = all frames; N = N/split dry-run
OUTPUT_DIR  = Path('outputs/lentils_concrete_run')             # inference notebook reads this (local mode)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Epochs: {MAX_EPOCHS} | image_size: {IMAGE_SIZE} | smoke_limit: {SMOKE_LIMIT or "(all)"}')

## 2 · Get + prepare the data

A splits CSV of `(split, npz_path, image_id)` that `MultiNpzDataModule` reads. `local` uses
the on-server NPZ directly.

In [ ]:
if config['data_source'] == 'local':
    SPLITS_CSV = resolve_splits_csv()
else:
    SPLITS_CSV = utils.ensure_lentils_npz(OUTPUT_DIR / 'npz', limit=SMOKE_LIMIT)  # raises until converter fix

if SMOKE_LIMIT and config['data_source'] == 'local':
    SPLITS_CSV = utils.subsample_splits_csv(SPLITS_CSV, SMOKE_LIMIT, OUTPUT_DIR / 'splits_smoke.csv')

print('Train splits CSV:', SPLITS_CSV)

### Peek at one frame

False-color RGB of a labelled frame with its ground-truth contour (the concrete selector
learns its own bands during training; this is just a human-readable preview of the scene).

In [ ]:
import csv
import matplotlib.pyplot as plt

rows = list(csv.DictReader(open(SPLITS_CSV)))
peek = next((r for r in rows if r.get('split') in ('test', 'val')), rows[0])
fr = utils.load_lentils_frame(peek['npz_path'])
view = utils.false_color(fr['cube'], fr['wavelengths'], (650.0, 550.0, 450.0))
present = sorted({int(c) for c in np.unique(fr['class_mask']) if int(c) != 0})
labels = [LENTILS_CATEGORIES.get(c, str(c)) for c in present]
print('frame:', Path(peek['npz_path']).name, '| classes present:', labels or '(normal)')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(view); ax[0].set_title('false-color RGB'); ax[0].axis('off')
ax[1].imshow(view)
if fr['mask'].any():
    ax[1].contour(fr['mask'] > 0, levels=[0.5], colors='red', linewidths=1.2)
ax[1].set_title('scene + GT contour'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 3 · Build the pipeline + distinctness regulariser

```
LentilsAnomalyDataNode -> MinMaxNormalizer -> ConcreteChannelMixer(61 -> 3)
     -> DinomalyDetector(input_channels=3) -> {QuantileBinaryDecider, AnomalyDetectionMetrics,
        AnomalyAUROCMetrics} -> TensorBoardMonitorNode
                            \-> selection_weights -> TrainOnlyDistinctnessLoss
```

`TrainOnlyDistinctnessLoss` is a tiny train-only node: it penalises pairwise cosine
similarity between the selector's output-channel weight vectors, so the 3 learned channels
stay spectrally distinct instead of collapsing onto one band. It is defined inline below
(it lives in the standalone `train_dinomaly_concrete_joint_multifile.py`, not the plugin).

In [ ]:
import torch.nn.functional as F
from torch import Tensor
from typing import Any
from cuvis_ai_core.node import Node
from cuvis_ai_schemas.enums import ExecutionStage
from cuvis_ai_schemas.pipeline import PortSpec


class TrainOnlyDistinctnessLoss(Node):
    """Pairwise cosine repulsion loss for selector weights, train stage only."""

    INPUT_SPECS = {
        'selection_weights': PortSpec(dtype=torch.float32, shape=(-1, -1),
                                      description='Selector weights [C_out, C_in].')
    }
    OUTPUT_SPECS = {
        'loss': PortSpec(dtype=torch.float32, shape=(), description='Distinctness loss')
    }

    def __init__(self, weight: float = 0.1, eps: float = 1e-6, **kwargs: Any) -> None:
        self.weight = float(weight)
        self.eps = float(eps)
        super().__init__(execution_stages={ExecutionStage.TRAIN},
                         weight=self.weight, eps=self.eps, **kwargs)

    def forward(self, selection_weights: Tensor, **_: Any) -> dict:
        w_norm = F.normalize(selection_weights, p=2, dim=-1, eps=self.eps)
        sim = w_norm @ w_norm.T
        upper = torch.triu(sim, diagonal=1)
        vals = upper[upper != 0]
        if vals.numel() == 0:
            loss = torch.zeros((), device=w_norm.device, dtype=w_norm.dtype)
        else:
            loss = vals.mean()
        return {'loss': self.weight * loss}

In [ ]:
from cuvis_ai.node.data import LentilsAnomalyDataNode
from cuvis_ai.node.normalization import MinMaxNormalizer
from cuvis_ai.node.channel_mixer import ConcreteChannelMixer
from cuvis_ai.node.metrics import AnomalyDetectionMetrics
from cuvis_ai.node.monitor import TensorBoardMonitorNode
from cuvis_ai.deciders.binary_decider import QuantileBinaryDecider
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_dinomaly.node.dinomaly_detector import DinomalyDetector
from cuvis_ai_dinomaly.node.dinomaly_train_loss_bridge import DinomalyTrainLossBridge
from cuvis_ai_dinomaly.node.auroc_metrics import AnomalyAUROCMetrics

pl.seed_everything(42, workers=True)

datamodule = MultiNpzDataModule(splits_csv=str(SPLITS_CSV), batch_size=1, num_workers=0)
datamodule.setup(stage='fit')
input_channels = int(next(iter(datamodule.train_dataloader()))['cube'].shape[-1])
print('cube channels ->', input_channels)

pipeline = CuvisPipeline('dinomaly_lentils_concrete')
data_node  = LentilsAnomalyDataNode(normal_class_ids=[0])
normalizer = MinMaxNormalizer(eps=1e-6, use_running_stats=True, max_initialization_frames=20)
selector   = ConcreteChannelMixer(
    input_channels=input_channels, output_channels=3, tau_start=10.0, tau_end=0.1,
    max_epochs=MAX_EPOCHS, use_hard_inference=True, eps=1e-6, name='concrete_selector')
dinomaly   = DinomalyDetector(
    encoder_name='dinov2reg_vit_base_14', bottleneck_dropout=0.2, decoder_depth=8,
    image_size=IMAGE_SIZE, crop_size=IMAGE_SIZE, use_center_crop=False,
    input_channels=3, name='dinomaly_detector')
loss_bridge  = DinomalyTrainLossBridge(weight=1.0, name='dinomaly_train_loss')
distinctness = TrainOnlyDistinctnessLoss(weight=0.1, eps=1e-6, name='distinctness_loss')
decider      = QuantileBinaryDecider(quantile=0.995, name='decider')
metrics_node = AnomalyDetectionMetrics(name='metrics_anomaly')
auroc_node   = AnomalyAUROCMetrics(name='metrics_auroc')
tb           = TensorBoardMonitorNode(output_dir=str(OUTPUT_DIR / 'tensorboard'), run_name=pipeline.name)

pipeline.connect(
    (data_node.outputs.cube,            normalizer.data),
    (normalizer.normalized,             selector.data),
    (selector.rgb,                      dinomaly.rgb_image),
    (dinomaly.outputs.training_loss,    loss_bridge.raw_loss),
    (selector.selection_weights,        distinctness.selection_weights),
    (dinomaly.outputs.scores,           decider.logits),
    (dinomaly.outputs.scores,           metrics_node.logits),
    (decider.decisions,                 metrics_node.decisions),
    (data_node.outputs.mask,            metrics_node.targets),
    (metrics_node.metrics,              tb.metrics),
    (dinomaly.outputs.scores,           auroc_node.scores),
    (data_node.outputs.mask,            auroc_node.targets),
    (dinomaly.outputs.anomaly_score,    auroc_node.anomaly_score),
)
print('Pipeline:', [n.name for n in pipeline.nodes])

## 4 · Train (joint)

`StatisticalTrainer` initialises the normaliser, then `GradientTrainer` trains **both** the
concrete selector and the Dinomaly bottleneck + decoder, with **two** loss nodes: the
reconstruction loss bridge and the distinctness regulariser. Optimizer matches
`configs/trainrun/dinomaly_multifile_concrete_joint.yaml` (AdamW, lr 2e-3).

In [ ]:
from cuvis_ai_core.training import GradientTrainer, StatisticalTrainer
from cuvis_ai_core.training.config import create_callbacks_from_config
from cuvis_ai_schemas.training import (
    CallbacksConfig, ModelCheckpointConfig, OptimizerConfig, TrainerConfig,
)

trainer_cfg = TrainerConfig(
    max_epochs=MAX_EPOCHS, accelerator='auto', devices=1,
    default_root_dir=str(OUTPUT_DIR), precision='32-true',
    enable_progress_bar=True, enable_checkpointing=True, log_every_n_steps=10,
    check_val_every_n_epoch=1, gradient_clip_val=0.1,
    callbacks=CallbacksConfig(checkpoint=ModelCheckpointConfig(
        dirpath=str(OUTPUT_DIR / 'checkpoints'), monitor='metrics_anomaly/iou',
        mode='max', save_top_k=1, save_last=True, filename='{epoch:02d}')),
)
optimizer_cfg = OptimizerConfig(name='adamw', lr=2e-3, weight_decay=1e-4, betas=[0.9, 0.999])

if normalizer.requires_initial_fit:
    print('Phase 1: statistical initialization (MinMaxNormalizer)...')
    StatisticalTrainer(pipeline=pipeline, datamodule=datamodule).fit()

pipeline.unfreeze_nodes_by_name(['dinomaly_detector', 'concrete_selector'])
pipeline.to(DEVICE)
n_train = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in pipeline.parameters())
print(f'Trainable: {n_train:,} / {n_total:,} ({100*n_train/max(n_total,1):.2f}%)')

callbacks = list(create_callbacks_from_config(trainer_cfg.callbacks)) if trainer_cfg.callbacks else []
print(f'Phase 2: joint gradient training for {MAX_EPOCHS} epoch(s)...')
grad_trainer = GradientTrainer(
    pipeline=pipeline, datamodule=datamodule,
    loss_nodes=[loss_bridge, distinctness], metric_nodes=[metrics_node, auroc_node],
    trainer_config=trainer_cfg, optimizer_config=optimizer_cfg,
    monitors=[tb], callbacks=callbacks,
)
grad_trainer.fit()
print('Training done.')

## 5 · Save the trained pipeline

Writes `<OUTPUT_DIR>/trained_models/dinomaly_lentils_concrete.yaml` + `.pt`. Point the
inference notebook at it with `LENTILS_PIPELINE_DIR=<this dir>`.

In [ ]:
from cuvis_ai_schemas.pipeline import PipelineMetadata

results_dir = OUTPUT_DIR / 'trained_models'
results_dir.mkdir(parents=True, exist_ok=True)
pipeline_path = results_dir / f'{pipeline.name}.yaml'
pipeline.save_to_file(
    str(pipeline_path),
    metadata=PipelineMetadata(
        name=pipeline.name,
        description='Dinomaly on lentils VNIR NPZ, Concrete learnable band selector (61->3) with distinctness regulariser.',
        tags=['dinomaly', 'anomalib', 'lentils', 'concrete', 'hyperspectral'],
        author='cuvis.ai'),
)
print('Saved:', pipeline_path)
pt = pipeline_path.with_suffix('.pt')
print('Weights:', pt, f'({pt.stat().st_size/1e6:.0f} MB)' if pt.is_file() else '(missing)')

## 6 · Next

Evaluate with **`lentils_inference_tutorial.ipynb`** and
`LENTILS_PIPELINE_DIR=notebooks/lentils_anomaly/outputs/lentils_concrete_run/trained_models`
for the per-class AUROC breakdown. For the full run: `LENTILS_MAX_EPOCHS=50` here, or
`uv run python examples/train_dinomaly_concrete_joint_multifile.py data.splits_csv=<your CSV>`.

> Remember the `tau` anneal schedule is tied to `MAX_EPOCHS` — a real learned band selection
> needs the full training length, not a 1-epoch smoke.